In [3]:
import pandas as pd

df = pd.read_csv('survey_results_public.csv', low_memory=False)
schema = pd.read_csv('survey_results_schema.csv')

df.head()


,ResponseId,MainBranch,Age,EdLevel,Employment,EmploymentAddl,WorkExp,LearnCodeChoose,LearnCode,LearnCodeAI,...,AIAgentOrchestration,AIAgentOrchWrite,AIAgentObserveSecure,AIAgentObsWrite,AIAgentExternal,AIAgentExtWrite,AIHuman,AIOpen,ConvertedCompYearly,JobSat
0,1,I am a developer by profession,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,"Caring for dependents (children, elderly, etc.)",8.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,Vertex AI,NaN,NaN,NaN,ChatGPT,NaN,When I don’t trust AI’s answers,"Troubleshooting, profiling, debugging",61256.0,10.0
1,2,I am a developer by profession,25-34 years old,"Associate degree (A.A., A.S., etc.)",Employed,NaN,2.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers;When I want to...,All skills. AI is a flop.,104413.0,9.0
2,3,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Independent contractor, freelancer, or self-em...",None of the above,10.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,ChatGPT;Claude Code;GitHub Copilot;Google Gemini,NaN,When I don’t trust AI’s answers;When I want to...,"Understand how things actually work, problem s...",53061.0,8.0
3,4,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,None of the above,4.0,"Yes, I am not new to coding but am learning ne...","Other online resources (e.g. standard search, ...","Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,ChatGPT;Claude Code,NaN,When I don’t trust AI’s answers;When I want to...,NaN,36197.0,6.0
4,5,I am a developer by profession,35-44 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Independent contractor, freelancer, or self-em...","Caring for dependents (children, elderly, etc.)",21.0,"No, I am not new to coding and did not learn n...",NaN,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers,"critical thinking, the skill to define the tas...",60000.0,7.0


In [4]:
#Кількість респондентів у датасеті
total_respondents = df.shape[0]
total_respondents

49191

In [5]:
#Отримуємо список питань зі schema
questions = set(schema['qname'])
#Залишаємо тільки ті, що реально є в основному датасеті
existing_questions = questions.intersection(set(df.columns))
len(existing_questions)


126

In [6]:
#Вибираємо тільки колонки з питаннями
df_filtered = df[list(existing_questions)]
#Підрахунок респондентів без жодного пропуску
complete_responses = df_filtered.dropna().shape[0]
complete_responses

0

In [7]:
#Перетворюємо WorkExp у числовий формат
df['WorkExp'] = pd.to_numeric(df['WorkExp'], errors='coerce')
# Видаляємо пропущені значення
work_exp = df['WorkExp'].dropna()
#Обчислення статистик
mean_exp = work_exp.mean()
median_exp = work_exp.median()
mode_exp = work_exp.mode()[0]
mean_exp, median_exp, mode_exp


(np.float64(13.367402606485907), 10.0, np.float64(10.0))

In [9]:
#перевірка,які значення має колонка RemoteWork.
df['RemoteWork'].value_counts()

RemoteWork
Remote                                                                          10931
Hybrid (some remote, leans heavy to in-person)                                   6732
In-person                                                                        6042
Hybrid (some in-person, leans heavy to flexibility)                              5831
Your choice (very flexible, you can come in when you want or just as needed)     4244
Name: count, dtype: int64

In [10]:
#Кількість респондентів,які працюють повністю віддалено
remote_count = df[df['RemoteWork'] == 'Remote'].shape[0]
remote_count

10931

In [24]:
#Перевіряємо, хто використовує Python
python_users = df['LanguageHaveWorkedWith'].str.contains('Python', na=False)
#Обчислюємо відсоток від усіх респондентів
python_percentage = python_users.mean() * 100
round(python_percentage, 2)

np.float64(37.54)

In [23]:
df['LearnCode'].value_counts().head(10)

LearnCode
Other online resources (e.g. standard search, forum, online community);Stack Overflow or Stack Exchange;Technical documentation (is generated for/by the tool or system)                                                                                                                               632
Other online resources (e.g. standard search, forum, online community);Technical documentation (is generated for/by the tool or system)                                                                                                                                                                465
Online Courses or Certification (includes all media types)                                                                                                                                                                                                                                             405
Other online resources (e.g. standard search, forum, online community);Stack Overflow or Stac

In [13]:
#Видаляємо пропуски
learn_code = df['LearnCode'].dropna()
#Перевіряємо, хто навчався через онлайн курси
online_learners = learn_code.str.contains(
    'Online Courses or Certification',
    na=False
)
#Кількість респондентів
online_learners_count = online_learners.sum()
online_learners_count


np.int64(10973)

In [15]:
#Перетворюємо компенсацію у числовий формат
df['ConvertedCompYearly'] = pd.to_numeric(
    df['ConvertedCompYearly'],
    errors='coerce'
)
#Фільтрація Python-розробників з наявною компенсацією
python_df = df[
    (df['LanguageHaveWorkedWith'].str.contains('Python', na=False)) &
    (df['ConvertedCompYearly'].notna())
]
#Групування по країнах
country_stats = (
    python_df
    .groupby('Country')['ConvertedCompYearly']
    .agg(['mean', 'median'])
    .reset_index()
)
country_stats.head()


,Country,mean,median
0,Afghanistan,22328.666667,1000.0
1,Albania,47217.600000,50000.0
2,Algeria,20187.285714,7088.0
3,Andorra,226103.500000,226103.5
4,Antigua and Barbuda,1.000000,1.0


In [17]:
#Видаляємо пропуски в компенсації
df_valid_comp = df[df['ConvertedCompYearly'].notna()]
#Сортуємо у спадному порядку
top5 = df_valid_comp.sort_values(
    by='ConvertedCompYearly',
    ascending=False
).head(5)
#Виводимо компенсацію та рівень освіти
top5[['ConvertedCompYearly', 'EdLevel']]


,ConvertedCompYearly,EdLevel
34267,50000000.0,"Associate degree (A.A., A.S., etc.)"
28700,33552715.0,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)"
43143,18387548.0,"Associate degree (A.A., A.S., etc.)"
35353,15430267.0,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)"
45971,13921760.0,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)"


In [19]:
# Видаляємо пропуски по Age
df_age = df[df['Age'].notna()]
# Загальна кількість респондентів по вікових категоріях
total_by_age = df_age.groupby('Age').size()
# Кількість Python-розробників по віку
python_by_age = (
    df_age[
        df_age['LanguageHaveWorkedWith'].str.contains('Python', na=False)
    ]
    .groupby('Age')
    .size()
)
#Обчислення відсотка
python_percentage_by_age = (
    (python_by_age / total_by_age) * 100
).reset_index(name='Python_Percentage')
python_percentage_by_age


,Age,Python_Percentage
0,18-24 years old,40.000000
1,25-34 years old,36.939282
2,35-44 years old,36.719281
3,45-54 years old,38.629482
4,55-64 years old,37.242955
5,65 years or older,31.634820
6,Prefer not to say,31.216931


In [21]:
#75-й перцентиль компенсації
percentile_75 = df['ConvertedCompYearly'].quantile(0.75)
#Фільтрація високооплачуваних віддалених працівників
high_paid_remote = df[
    (df['ConvertedCompYearly'] > percentile_75) &
    (df['RemoteWork'] == 'Remote')
]
#Підрахунок індустрій
industry_counts = (
    high_paid_remote['Industry']
    .value_counts()
)
industry_counts


Industry
Software Development                          1186
Fintech                                        190
Healthcare                                     188
Other:                                         176
Internet, Telecomm or Information Services     138
Banking/Financial Services                      88
Government                                      78
Media & Advertising Services                    75
Retail and Consumer Services                    65
Transportation, or Supply Chain                 63
Computer Systems Design and Services            59
Insurance                                       46
Energy                                          45
Manufacturing                                   33
Higher Education                                30
Name: count, dtype: int64